# Task 1

In [1]:
import pandas as pd
import numpy as np

df_r = pd.read_excel("Online Retail.xlsx")
df = df_r.copy()
print(f"Shape: {df.shape}")
print(f"\nColumn types:\n{df.dtypes}")
print(f"\nMissing values:\n{df.isnull().sum()}")
print(f"\nFirst rows:\n{df.head()}")

Shape: (541909, 8)

Column types:
InvoiceNo              object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[ns]
UnitPrice             float64
CustomerID            float64
Country                object
dtype: object

Missing values:
InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

First rows:
  InvoiceNo StockCode                          Description  Quantity  \
0    536365    85123A   WHITE HANGING HEART T-LIGHT HOLDER         6   
1    536365     71053                  WHITE METAL LANTERN         6   
2    536365    84406B       CREAM CUPID HEARTS COAT HANGER         8   
3    536365    84029G  KNITTED UNION FLAG HOT WATER BOTTLE         6   
4    536365    84029E       RED WOOLLY HOTTIE WHITE HEART.         6   

          InvoiceDate  UnitPrice  CustomerID         Country

In [2]:
number_of_row = df.shape[0]
number_of_columns = df.shape[1]
m_size = df.memory_usage(deep=True).sum()
print(number_of_row)
print(number_of_columns)
print(m_size)

541909
8
132310571


In [3]:
missing_value_count = df.isna().sum()
ratio = (df.isnull().sum() / df.shape[0] * 100)
print(ratio) 
print(missing_value_count)

InvoiceNo       0.000000
StockCode       0.000000
Description     0.268311
Quantity        0.000000
InvoiceDate     0.000000
UnitPrice       0.000000
CustomerID     24.926694
Country         0.000000
dtype: float64
InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64


In [4]:
# Number of unique values per column
number_unique_val_column = {}
for i in df.columns:
    number_unique_val_column[i] = df[i].nunique()
number_unique_val_column

{'InvoiceNo': 25900,
 'StockCode': 4070,
 'Description': 4223,
 'Quantity': 722,
 'InvoiceDate': 23260,
 'UnitPrice': 1630,
 'CustomerID': 4372,
 'Country': 38}

In [5]:
df.describe().T

,count,mean,min,25%,50%,75%,max,std
Quantity,541909.0,9.55225,-80995.0,1.0,3.0,10.0,80995.0,218.081158
InvoiceDate,541909,2011-07-04 13:34:57.156386048,2010-12-01 08:26:00,2011-03-28 11:34:00,2011-07-19 17:17:00,2011-10-19 11:27:00,2011-12-09 12:50:00,NaN
UnitPrice,541909.0,4.611114,-11062.06,1.25,2.08,4.13,38970.0,96.759853
CustomerID,406829.0,15287.69057,12346.0,13953.0,15152.0,16791.0,18287.0,1713.600303


In [6]:
df.groupby(df['CustomerID'].isna())[['Quantity','UnitPrice']].mean()

,Quantity,UnitPrice
CustomerID,,
False,12.061303,3.460471
True,1.995573,8.076577


In [7]:
with_na= df[df["CustomerID"].notna()]
without_na = df[df["CustomerID"].isna()]

print('With nan ',with_na['Country'].value_counts().head())
print('without nan ',without_na['Country'].value_counts().head())

With nan  Country
United Kingdom    361878
Germany             9495
France              8491
EIRE                7485
Spain               2533
Name: count, dtype: int64
without nan  Country
United Kingdom    133600
EIRE                 711
Hong Kong            288
Unspecified          202
Switzerland          125
Name: count, dtype: int64


I think that, this is MNAR(for customerid).Because this is directly depends on customer itself, when we look following result, we see that they but not too much units but price is high,
therefore we say that this is MNAR.In my opinion, we can delete them.

In [8]:
df.groupby(df['Description'].isna())[['Quantity','UnitPrice']].mean()

,Quantity,UnitPrice
Description,,
False,9.603129,4.623519
True,-9.359697,0.000000


In [9]:
most_common = (
    df.dropna(subset=["Description"])
    .groupby("StockCode")["Description"]
    .agg(lambda x: x.mode()[0])
)
most_common

StockCode
10002                  INFLATABLE POLITICAL GLOBE 
10080                     GROOVY CACTUS INFLATABLE
10120                                 DOGGY RUBBER
10125                      MINI FUNKY DESIGN TAPES
10133                 COLOURING PENCILS BROWN TUBE
                               ...                
gift_0001_20    Dotcomgiftshop Gift Voucher £20.00
gift_0001_30    Dotcomgiftshop Gift Voucher £30.00
gift_0001_40    Dotcomgiftshop Gift Voucher £40.00
gift_0001_50    Dotcomgiftshop Gift Voucher £50.00
m                                           Manual
Name: Description, Length: 3958, dtype: object

In [10]:
missing_desc = df[df['Description'].isna()]
missing_desc

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
622,536414,22139,NaN,56,2010-12-01 11:52:00,0.0,NaN,United Kingdom
1970,536545,21134,NaN,1,2010-12-01 14:32:00,0.0,NaN,United Kingdom
1971,536546,22145,NaN,1,2010-12-01 14:33:00,0.0,NaN,United Kingdom
1972,536547,37509,NaN,1,2010-12-01 14:33:00,0.0,NaN,United Kingdom
1987,536549,85226A,NaN,1,2010-12-01 14:34:00,0.0,NaN,United Kingdom
...,...,...,...,...,...,...,...,...
535322,581199,84581,NaN,-2,2011-12-07 18:26:00,0.0,NaN,United Kingdom
535326,581203,23406,NaN,15,2011-12-07 18:31:00,0.0,NaN,United Kingdom
535332,581209,21620,NaN,6,2011-12-07 18:35:00,0.0,NaN,United Kingdom
536981,581234,72817,NaN,27,2011-12-08 10:33:00,0.0,NaN,United Kingdom


In [11]:
recoverable = missing_desc["StockCode"].isin(most_common.index).sum()
recoverable

1342

I think this is MAR(for description). Soo, i will delete description because , when we look at the result of below code s we see that unitprice is 0 for all and customer id is not found also therefore we can delete this nans

In [12]:
print(f"Before dropping missing CustomerID: {len(df):,} rows")
df = df.dropna(subset=["CustomerID"])
print(f"After  dropping missing CustomerID: {len(df):,} rows")

Before dropping missing CustomerID: 541,909 rows
After  dropping missing CustomerID: 406,829 rows


In [13]:
(missing_desc['CustomerID'].isna()).all()

True

In [14]:
(missing_desc['UnitPrice']==0).all()

True

In [15]:
df = df[df['Description'].notna()].copy()
print(df.isnull().sum())

InvoiceNo      0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
UnitPrice      0
CustomerID     0
Country        0
dtype: int64


In [16]:
df

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
...,...,...,...,...,...,...,...,...
541904,581587,22613,PACK OF 20 SPACEBOY NAPKINS,12,2011-12-09 12:50:00,0.85,12680.0,France
541905,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,2011-12-09 12:50:00,2.10,12680.0,France
541906,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,2011-12-09 12:50:00,4.15,12680.0,France
541907,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,2011-12-09 12:50:00,4.15,12680.0,France


# Task 2

In [17]:
cancellations = df[df['InvoiceNo'].astype(str).str.startswith('C')]
print(cancellations)
print(len(cancellations))

       InvoiceNo StockCode                       Description  Quantity  \
141      C536379         D                          Discount        -1   
154      C536383    35004C   SET OF 3 COLOURED  FLYING DUCKS        -1   
235      C536391     22556    PLASTERS IN TIN CIRCUS PARADE        -12   
236      C536391     21984  PACK OF 12 PINK PAISLEY TISSUES        -24   
237      C536391     21983  PACK OF 12 BLUE PAISLEY TISSUES        -24   
...          ...       ...                               ...       ...   
540449   C581490     23144   ZINC T-LIGHT HOLDER STARS SMALL       -11   
541541   C581499         M                            Manual        -1   
541715   C581568     21258        VICTORIAN SEWING BOX LARGE        -5   
541716   C581569     84978  HANGING HEART JAR T-LIGHT HOLDER        -1   
541717   C581569     20979     36 PENCILS TUBE RED RETROSPOT        -5   

               InvoiceDate  UnitPrice  CustomerID         Country  
141    2010-12-01 09:41:00      27.50     1

In [18]:
df['is_cancelled'] = df['InvoiceNo'].astype(str).str.startswith('C').astype(int)

In [19]:
invalid_q = df[(df['is_cancelled'] == 0) & (df['Quantity'] <= 0)]
invalid_price = df[df['UnitPrice'] <= 0]
print(len(invalid_q))
print(len(invalid_price))

0
40


In [20]:
df[df['is_cancelled']==0]['Quantity'].describe(percentiles=[.25,.75,.95,.99])

count    397924.000000
mean         13.021823
std         180.420210
min           1.000000
25%           2.000000
50%           6.000000
75%          12.000000
95%          36.000000
99%         120.000000
max       80995.000000
Name: Quantity, dtype: float64

In [21]:
df[df['UnitPrice'] > 0]['UnitPrice'].describe(percentiles=[.25,.75,.95,.99])

count    406789.000000
mean          3.460811
std          69.318561
min           0.001000
25%           1.250000
50%           1.950000
75%           3.750000
95%           8.500000
99%          15.000000
max       38970.000000
Name: UnitPrice, dtype: float64

In [22]:
def iqr_bounds(series):
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    return Q1 - 1.5 * IQR, Q3 + 1.5 * IQR

qty_clean = df[df['is_cancelled']==0]['Quantity']
price_clean = df[df['UnitPrice'] > 0]['UnitPrice']

qty_lower, qty_upper     = iqr_bounds(qty_clean)
price_lower, price_upper = iqr_bounds(price_clean)

print(f"Quantity outlier lower={qty_lower:.1f},   upper={qty_upper:.1f}")
print(f"UnitPrice outlier lower={price_lower:.2f}, upper={price_upper:.2f}")

qty_outliers   = qty_clean[(qty_clean < qty_lower) | (qty_clean > qty_upper)]
price_outliers = price_clean[(price_clean < price_lower) | (price_clean > price_upper)]

print(f"Quantity  outliers: {len(qty_outliers):,}")
print(f"UnitPrice outlies: {len(price_outliers):,}")

Quantity outlier lower=-13.0,   upper=27.0
UnitPrice outlier lower=-2.50, upper=7.50
Quantity  outliers: 25,656
UnitPrice outlies: 36,051


In [23]:
df = df[~((df['is_cancelled'] == 0) & (df['Quantity'] <= 0))].copy()
df = df[df['UnitPrice'] > 0].copy()
qty_cap = df[df['is_cancelled'] == 0]['Quantity'].quantile(0.99)
price_cap = df['UnitPrice'].quantile(0.99)
mask = df['is_cancelled'] == 0
df.loc[mask, 'Quantity'] = df.loc[mask, 'Quantity'].clip(upper=qty_cap)
df['UnitPrice'] = df['UnitPrice'].clip(upper=price_cap)
print(f"Təmizlənmiş dataset: {df.shape}")
print(df.isnull().sum())

Təmizlənmiş dataset: (406789, 9)
InvoiceNo       0
StockCode       0
Description     0
Quantity        0
InvoiceDate     0
UnitPrice       0
CustomerID      0
Country         0
is_cancelled    0
dtype: int64


Instead of removing outliers, I decided to cap them at the 99th percentile because these high amounts could be real but rare wholesale transactions and I don't want to lose the data completely.

In [24]:
print(f'Shape before cleaning: {df_r.shape}')
print(f"Shape after cleaning: {df.shape}")
print(((df['is_cancelled']==0) & (df['Quantity'] <= 0)).sum())
print((df['UnitPrice'] <= 0).sum())
print(df['Quantity'].min())
print(df['UnitPrice'].min())

Shape before cleaning: (541909, 8)
Shape after cleaning: (406789, 9)
0
0
-80995
0.001


# Task 3

In [25]:
uniq_countries = df['Country'].nunique()
uniq_countries

37

In [26]:
country_counts = df['Country'].value_counts()
top5_pct = country_counts.head(5) / len(df) * 100
print(top5_pct)

Country
United Kingdom    88.953733
Germany            2.333642
France             2.087077
EIRE               1.839529
Spain              0.622436
Name: count, dtype: float64


In [27]:
country_counts = df['Country'].value_counts()
rare_countries = country_counts[country_counts < 50].index
df['Country_cleaned'] = df['Country'].apply(
    lambda x: 'Other' if x in rare_countries else x
)

In [28]:
print(country_counts)
print(rare_countries)
df

Country
United Kingdom          361854
Germany                   9493
France                    8490
EIRE                      7483
Spain                     2532
Netherlands               2367
Belgium                   2069
Switzerland               1876
Portugal                  1480
Australia                 1256
Norway                    1085
Italy                      803
Channel Islands            758
Finland                    695
Cyprus                     622
Sweden                     462
Austria                    401
Denmark                    389
Japan                      358
Poland                     341
USA                        291
Israel                     250
Unspecified                244
Singapore                  229
Iceland                    182
Canada                     151
Greece                     146
Malta                      127
United Arab Emirates        68
European Community          61
RSA                         57
Lebanon                     45


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,is_cancelled,Country_cleaned
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,0,United Kingdom
...,...,...,...,...,...,...,...,...,...,...
541904,581587,22613,PACK OF 20 SPACEBOY NAPKINS,12,2011-12-09 12:50:00,0.85,12680.0,France,0,France
541905,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,2011-12-09 12:50:00,2.10,12680.0,France,0,France
541906,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,2011-12-09 12:50:00,4.15,12680.0,France,0,France
541907,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,2011-12-09 12:50:00,4.15,12680.0,France,0,France


In [29]:
country_counts = df['Country'].value_counts()
rare_mask = country_counts < 50
print(f"Rare countries (<50 transactions): {rare_mask.sum()}")
print(f"Before cleaning: {uniq_countries} categories")
print(f"After cleaning:  {df['Country_cleaned'].nunique()} categories")

Rare countries (<50 transactions): 6
Before cleaning: 37 categories
After cleaning:  32 categories


In [30]:
print('Unique stock codes: ',df['StockCode'].nunique())

Unique stock codes:  3684


In [31]:
df['StockCode'] = df['StockCode'].astype(str)

In [32]:
numeric_mask = df['StockCode'].str.match(r'^\d+$')
non_numeric = df[~numeric_mask]['StockCode'].unique()
print(len(non_numeric))
print(sorted(non_numeric))

886
['10123C', '10124A', '10124G', '15044A', '15044B', '15044C', '15044D', '15056BL', '15056N', '15056P', '15058A', '15058B', '15058C', '15060B', '16020C', '16151A', '16156L', '16156S', '16161G', '16161M', '16161P', '16161U', '16162L', '16162M', '16168M', '16169E', '16169K', '16169M', '16169N', '16169P', '16202A', '16202B', '16202E', '16206B', '16207A', '16207B', '16244B', '16248B', '16258A', '17007B', '17011F', '17012A', '17012B', '17012C', '17012D', '17012E', '17012F', '17013D', '17014A', '17028J', '17084A', '17084J', '17084N', '17084P', '17084R', '17090A', '17090D', '17091A', '17091J', '17107D', '17109D', '17129F', '17136A', '17164B', '17165D', '17191A', '18094C', '18097A', '18097C', '18098C', '35001G', '35001W', '35004B', '35004C', '35004G', '35004P', '35095A', '35095B', '35271S', '35471D', '35591T', '35597A', '35597B', '35597D', '35598A', '35598B', '35598C', '35598D', '35599B', '35599D', '35607A', '35607B', '35610A', '35610B', '35610C', '35637A', '35637C', '35638A', '35638B', '358

In [33]:
non_product_mask = ~df['StockCode'].str.match(r'^\d+[A-Z]?$')
print(f"Non-product rows: {non_product_mask.sum()}")
df = df[~non_product_mask].copy()
print(f"Shape after removing non-products: {df.shape}")

Non-product rows: 2209
Shape after removing non-products: (404580, 10)


In [34]:
df

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,is_cancelled,Country_cleaned
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,0,United Kingdom
...,...,...,...,...,...,...,...,...,...,...
541904,581587,22613,PACK OF 20 SPACEBOY NAPKINS,12,2011-12-09 12:50:00,0.85,12680.0,France,0,France
541905,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,2011-12-09 12:50:00,2.10,12680.0,France,0,France
541906,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,2011-12-09 12:50:00,4.15,12680.0,France,0,France
541907,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,2011-12-09 12:50:00,4.15,12680.0,France,0,France


In [35]:
df['Description'].nunique()

3887

In [36]:
df['desc_word_count'] = df['Description'].str.split().str.len()

In [37]:
df

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,is_cancelled,Country_cleaned,desc_word_count
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,0,United Kingdom,5
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,0,United Kingdom,3
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,0,United Kingdom,5
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,0,United Kingdom,6
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,0,United Kingdom,5
...,...,...,...,...,...,...,...,...,...,...,...
541904,581587,22613,PACK OF 20 SPACEBOY NAPKINS,12,2011-12-09 12:50:00,0.85,12680.0,France,0,France,5
541905,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,2011-12-09 12:50:00,2.10,12680.0,France,0,France,4
541906,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,2011-12-09 12:50:00,4.15,12680.0,France,0,France,4
541907,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,2011-12-09 12:50:00,4.15,12680.0,France,0,France,4


In [38]:
print(df.groupby('desc_word_count')['UnitPrice'].mean())
print(df.groupby('desc_word_count')['Quantity'].mean())

desc_word_count
1    1.807627
2    3.826871
3    2.631509
4    2.927909
5    2.732654
6    2.619020
7    3.066703
8    1.531243
Name: UnitPrice, dtype: float64
desc_word_count
1    25.118644
2    11.198563
3    11.517101
4    10.209461
5     8.707940
6    10.223677
7     9.818460
8    12.205026
Name: Quantity, dtype: float64


# Task 4

In [39]:
df['revenue'] = df['Quantity'] * df['UnitPrice']
customer_df = df.groupby('CustomerID').agg(
    total_revenue  = ('revenue', 'sum'),
    num_orders     = ('InvoiceNo', 'nunique'),
    num_products   = ('StockCode', 'nunique'),
    first_purchase = ('InvoiceDate', 'min'),
    last_purchase  = ('InvoiceDate', 'max')
).reset_index()
threshold = customer_df['total_revenue'].quantile(0.90)
customer_df['is_high_value'] = (customer_df['total_revenue'] >= threshold).astype(int)
threshold

3365.194

In [40]:
customer_df

,CustomerID,total_revenue,num_orders,num_products,first_purchase,last_purchase,is_high_value
0,12346.0,-77058.80,2,1,2011-01-18 10:01:00,2011-01-18 10:17:00,0
1,12347.0,4185.20,7,103,2010-12-07 14:57:00,2011-12-07 15:52:00,1
2,12348.0,1395.48,4,21,2010-12-16 19:09:00,2011-09-25 13:13:00,0
3,12349.0,1428.80,1,72,2011-11-21 09:51:00,2011-11-21 09:51:00,0
4,12350.0,294.40,1,16,2011-02-02 16:01:00,2011-02-02 16:01:00,0
...,...,...,...,...,...,...,...
4357,18280.0,180.60,1,10,2011-03-07 09:52:00,2011-03-07 09:52:00,0
4358,18281.0,76.92,1,7,2011-06-12 10:53:00,2011-06-12 10:53:00,0
4359,18282.0,176.60,3,12,2011-08-05 13:35:00,2011-12-02 11:43:00,0
4360,18283.0,2087.98,16,262,2011-01-06 14:14:00,2011-12-06 12:02:00,0


In [41]:
dist = customer_df['is_high_value'].value_counts()

In [42]:
accuracy = dist[0]/dist.sum()
accuracy
print(f'{accuracy:.1%}')

90.0%


- The model always says 'regular' and gets ~90% accuracy
- But it doesn't find any high-value customers
- Finding high-value customers is important to us

In [43]:
from sklearn.model_selection import train_test_split

features = ['num_orders', 'num_products']
X = customer_df[features]
y = customer_df['is_high_value']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [44]:
from imblearn.over_sampling import RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler

ros = RandomOverSampler(random_state=42)
X_over, y_over = ros.fit_resample(X_train, y_train)

rus = RandomUnderSampler(random_state=42)
X_under, y_under = rus.fit_resample(X_train, y_train)

print(f"Original:     {y_train.value_counts().to_dict()}")
print(f"Oversampled:  {y_over.value_counts().to_dict()}")
print(f"Undersampled: {y_under.value_counts().to_dict()}")

Original:     {0: 3132, 1: 357}
Oversampled:  {0: 3132, 1: 3132}
Undersampled: {0: 357, 1: 357}


In [45]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score, f1_score

def evaluate(model, X_test, y_test, name):
    y_pred = model.predict(X_test)
    print(f"\n{name}:")
    print(f"  Precision: {precision_score(y_test, y_pred):.2f}")
    print(f"  Recall:    {recall_score(y_test, y_pred):.2f}")
    print(f"  F1:        {f1_score(y_test, y_pred):.2f}")

model_orig = LogisticRegression().fit(X_train, y_train)
evaluate(model_orig, X_test, y_test, "Original")

model_over = LogisticRegression().fit(X_over, y_over)
evaluate(model_over, X_test, y_test, "Oversampled")

model_under = LogisticRegression().fit(X_under, y_under)
evaluate(model_under, X_test, y_test, "Undersampled")


Original:
  Precision: 0.75
  Recall:    0.54
  F1:        0.63

Oversampled:
  Precision: 0.46
  Recall:    0.85
  F1:        0.60

Undersampled:
  Precision: 0.48
  Recall:    0.85
  F1:        0.62


The original model has low recall — it misses out on high-value customers. Oversampling increases recall, but precision decreases slightly, but it gives a more balanced result. Undersampling gives the highest recall but precision is the lowest. It is more harmful for a business to miss out on high-value customers, so oversampling is more appropriate.

# Task 5

In [46]:
dec_2011 = df[df['InvoiceDate'].dt.to_period('M') == '2011-12']
dec_customers = set(dec_2011['CustomerID'].unique())


customer_df['bought_in_dec'] = customer_df['CustomerID'].isin(dec_customers).astype(int)
print("Target distribution:")
print(customer_df['bought_in_dec'].value_counts())

features = ['total_revenue', 'num_orders', 'num_products']
X = customer_df[features]
y = customer_df['bought_in_dec']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model_leaky = LogisticRegression().fit(X_train, y_train)
y_pred_leaky = model_leaky.predict(X_test)

print("\n Leaky model result:")
print(f"  Precision: {precision_score(y_test, y_pred_leaky):.2f}")
print(f"  Recall:    {recall_score(y_test, y_pred_leaky):.2f}")
print(f"  F1:        {f1_score(y_test, y_pred_leaky):.2f}")

Target distribution:
bought_in_dec
0    3678
1     684
Name: count, dtype: int64

 Leaky model result:
  Precision: 0.68
  Recall:    0.20
  F1:        0.31


Features are calculated over the entire history (Dec 2010 → Dec 2011). total_revenue also includes data from Dec 2011. Target is to make purchases in Dec 2011. So the model sees the answer in the feature — this is data leakage.

In [47]:
print(f"Feature data range: {df['InvoiceDate'].min()} → {df['InvoiceDate'].max()}")
correlations = customer_df[features + ['bought_in_dec']].corr()['bought_in_dec'].drop('bought_in_dec')
print("\nFeature-target correlations:")
print(correlations.sort_values(ascending=False))

Feature data range: 2010-12-01 08:26:00 → 2011-12-09 12:50:00

Feature-target correlations:
num_orders       0.348104
num_products     0.285988
total_revenue    0.180878
Name: bought_in_dec, dtype: float64


In [48]:
observation_end   = pd.Timestamp("2011-09-30")
prediction_start  = pd.Timestamp("2011-10-01")

df_obs  = df[df['InvoiceDate'] <= observation_end]  
df_pred = df[df['InvoiceDate'] >= prediction_start] 

customer_df_clean = df_obs.groupby('CustomerID').agg(
    total_revenue  = ('revenue', 'sum'),
    num_orders     = ('InvoiceNo', 'nunique'),
    num_products   = ('StockCode', 'nunique')
).reset_index()

pred_customers = set(df_pred['CustomerID'].unique())
customer_df_clean['will_buy'] = customer_df_clean['CustomerID'].isin(pred_customers).astype(int)

print("Target distribution (clean):")
print(customer_df_clean['will_buy'].value_counts())

X_clean = customer_df_clean[features]
y_clean = customer_df_clean['will_buy']

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_clean, y_clean, test_size=0.2, random_state=42
)

model_clean = LogisticRegression().fit(X_train_c, y_train_c)
y_pred_clean = model_clean.predict(X_test_c)

print("\n Clean model result:")
print(f"  Precision: {precision_score(y_test_c, y_pred_clean):.2f}")
print(f"  Recall:    {recall_score(y_test_c, y_pred_clean):.2f}")
print(f"  F1:        {f1_score(y_test_c, y_pred_clean):.2f}")

Target distribution (clean):
will_buy
1    1879
0    1757
Name: count, dtype: int64

 Clean model result:
  Precision: 0.76
  Recall:    0.60
  F1:        0.67


Features are only calculated up to Sep 2011, target is Oct-Dec 2011. Model doesn't see the future now. F1 is down — that's normal and correct. In real world, leaky model would not work at all in production because future information is not available.